In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [3]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-10-31 10:15:25.841726-04:00


In [ ]:
# Pull claim features + flags
q_claims = """
SELECT claimid, provider, beneid, claim_type, claim_start, claim_end,
       reimb_amt, deductible_paid, los_days, dx_count, px_count, drg
FROM mart.features_claim
"""
claims = pd.read_sql(q_claims, con=engine)

rcf = pd.read_sql("SELECT * FROM mart.rule_claim_flags", con=engine)

z = pd.read_sql(
    "SELECT claimid, z_ip_reimb, z_ip_los, z_op_reimb FROM mart.rule_claim_z", con=engine)

df = (claims
      .merge(rcf, on="claimid", how="left")
      .merge(z, on="claimid", how="left"))

df

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,...,drg,dup_exact_flag,dup_near_count,upcoding_ip_flag,upcoding_op_flag,overcharge_z_flag,overcharge_iqr_flag,z_ip_reimb,z_ip_los,z_op_reimb
0,CLM569367,PRV55455,BENE100014,OP,2009-09-08,2009-09-08,100.00,0.00,NaN,1,...,None,0,0,0,0,0,0,NaN,NaN,-0.30
1,CLM76080,PRV55659,BENE100014,IP,2009-11-15,2009-11-17,"3,000.00","1,068.00",2.00,6,...,181,0,0,0,0,0,0,-0.67,-0.67,NaN
2,CLM280761,PRV55832,BENE100016,OP,2009-04-02,2009-04-02,60.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.35
3,CLM174738,PRV55368,BENE100021,OP,2009-02-03,2009-02-03,50.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.38
4,CLM296629,PRV55209,BENE100040,OP,2009-04-10,2009-04-10,200.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558206,CLM195970,PRV51074,BENE99980,OP,2009-02-15,2009-02-15,40.00,0.00,NaN,1,...,None,0,0,0,0,0,0,NaN,NaN,-0.50
558207,CLM693942,PRV51117,BENE99980,OP,2009-11-22,2009-11-22,80.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.30
558208,CLM491509,PRV57605,BENE99981,OP,2009-07-26,2009-07-26,400.00,0.00,NaN,5,...,None,0,0,0,0,0,0,NaN,NaN,0.32
558209,CLM117724,PRV56218,BENE99987,OP,2009-01-03,2009-01-03,100.00,0.00,NaN,1,...,None,0,0,0,0,0,0,NaN,NaN,-0.30


In [5]:
def iqr_flags(s: pd.Series):
    x = s.dropna()
    q1, q3 = x.quantile([.25, .75])
    iqr = q3 - q1
    return (s < (q1 - 1.5*iqr)) | (s > (q3 + 1.5*iqr))


def modified_z_flags(s: pd.Series):
    x = s.dropna()
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0 or np.isnan(mad):
        return pd.Series(False, index=s.index)
    Mi = 0.6745 * (s - med) / mad
    return Mi.abs() > 3.5

In [6]:
df["iqr_outlier_reimb"] = df.groupby(
    "claim_type")["reimb_amt"].transform(iqr_flags)
df["mad_outlier_reimb"] = df.groupby(
    "claim_type")["reimb_amt"].transform(modified_z_flags)

#### 2 - `Duplicate` & `near-duplicate` claims

In [7]:
df["duplicate_suspect"] = (df["dup_exact_flag"].fillna(
    0).eq(1)) | (df["dup_near_count"].fillna(0) > 0)

##### 3 - `Short-stay`, `high-payment` inpatient claims

In [8]:
df["short_stay_flag_py"] = (df["claim_type"].eq(
    "IP")) & (df["los_days"].fillna(0) < 2)
df["short_stay_costly"] = df["short_stay_flag_py"] & (
    df["z_ip_reimb"].fillna(0) > 2.5)

##### 4 - `Missingness` & `odd` shapes

In [ ]:
null_checks = {
    "reimb_amt_null": df["reimb_amt"].isna().sum(),
    "ip_missing_drg": df.query("claim_type=='IP'")["drg"].isna().sum(),
    "ip_missing_dates": df.query("claim_type=='IP'")[["claim_start", "claim_end"]].isna().any(axis=1).sum(),
    "op_missing_dxcount": df.query("claim_type=='OP'")["dx_count"].isna().sum()
}
null_checks

{'reimb_amt_null': np.int64(0),
 'ip_missing_drg': np.int64(0),
 'ip_missing_dates': np.int64(0),
 'op_missing_dxcount': np.int64(0)}

##### 5 - Build `case packets`

In [ ]:
cols = ["claimid", "provider", "beneid", "claim_type", "claim_start", "claim_end",
        "reimb_amt", "deductible_paid", "los_days", "dx_count", "px_count", "drg",
        "dup_exact_flag", "dup_near_count", "z_ip_reimb", "z_ip_los", "z_op_reimb"]

pkt_outlier = df.loc[df["iqr_outlier_reimb"]
                     | df["mad_outlier_reimb"], cols].copy()
pkt_dupe = df.loc[df["duplicate_suspect"], cols].copy()
pkt_short = df.loc[df["short_stay_costly"], cols].copy()

cut = df.groupby("claim_type")["reimb_amt"].transform(
    lambda s: s.quantile(0.99))
pkt_toppaid = df.loc[df["reimb_amt"] >= cut, cols].copy()


def tag(df_, label):
    x = df_.copy()
    x["reason"] = label
    return x


case_packets = pd.concat([
    tag(pkt_outlier, "outlier_reimb_iqr_or_mad"),
    tag(pkt_dupe,    "duplicate_or_near_duplicate"),
    tag(pkt_short,   "short_stay_high_zpay_ip"),
    tag(pkt_toppaid, "top1pct_paid")
], ignore_index=True).drop_duplicates(subset=["claimid", "reason"])
case_packets.head(100)

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,px_count,drg,dup_exact_flag,dup_near_count,z_ip_reimb,z_ip_los,z_op_reimb,reason
0,CLM434931,PRV53005,BENE100146,OP,2009-06-25,2009-06-25,500.00,0.00,NaN,4,0,None,0,0,NaN,NaN,0.07,outlier_reimb_iqr_or_mad
1,CLM416648,PRV57434,BENE100154,OP,2009-06-15,2009-06-15,800.00,0.00,NaN,4,0,None,0,0,NaN,NaN,1.23,outlier_reimb_iqr_or_mad
2,CLM285505,PRV51860,BENE100180,OP,2009-04-04,2009-04-04,"1,300.00",0.00,NaN,3,0,None,0,0,NaN,NaN,1.81,outlier_reimb_iqr_or_mad
3,CLM234546,PRV55834,BENE100264,OP,2009-03-08,2009-03-08,"2,000.00",0.00,NaN,0,0,None,0,0,NaN,NaN,NaN,outlier_reimb_iqr_or_mad
4,CLM119588,PRV57191,BENE100299,OP,2009-01-04,2009-01-04,"1,200.00",0.00,NaN,3,0,None,0,0,NaN,NaN,1.62,outlier_reimb_iqr_or_mad
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CLM242895,PRV51250,BENE103859,OP,2009-03-12,2009-03-12,"2,300.00",0.00,NaN,1,0,None,0,0,NaN,NaN,3.00,outlier_reimb_iqr_or_mad
96,CLM724144,PRV56738,BENE103864,OP,2009-12-11,2009-12-11,800.00,0.00,NaN,4,0,None,0,0,NaN,NaN,1.23,outlier_reimb_iqr_or_mad
97,CLM137806,PRV54367,BENE103882,OP,2009-01-14,2009-01-14,"1,800.00",0.00,NaN,1,0,None,0,0,NaN,NaN,1.82,outlier_reimb_iqr_or_mad
98,CLM244753,PRV53773,BENE103901,OP,2009-03-13,2009-04-02,400.00,0.00,NaN,10,0,None,0,0,NaN,NaN,0.07,outlier_reimb_iqr_or_mad


##### Quick sanity counts

In [11]:
summary = pd.DataFrame({
    "outlier_reimb_iqr_or_mad": [len(pkt_outlier)],
    "duplicate_or_near_duplicate": [len(pkt_dupe)],
    "short_stay_high_zpay_ip": [len(pkt_short)],
    "top1pct_paid": [len(pkt_toppaid)],
})
summary.T.rename(columns={0: "n_claims"})

,n_claims
outlier_reimb_iqr_or_mad,101896
duplicate_or_near_duplicate,583
short_stay_high_zpay_ip,82
top1pct_paid,6439


##### Save packets for review

In [12]:
case_packets.to_csv("case_packets_claims.csv", index=False)

##### Derive the boolean flags

In [ ]:
q99 = df.groupby("claim_type")["reimb_amt"].transform(lambda s: s.quantile(0.99))
df["is_top1pct_paid"] = df["reimb_amt"] >= q99

df["peer_z_high"] = (
    (df["z_ip_reimb"].fillna(-np.inf) > 3) |
    (df["z_op_reimb"].fillna(-np.inf) > 3)
)

df["is_outlier_reimb"] = df["iqr_outlier_reimb"] | df["mad_outlier_reimb"]

df["duplicate_suspect"] = df["duplicate_suspect"].fillna(False)

for col in ["overcharge_z_flag","overcharge_iqr_flag","dup_exact_flag","dup_near_count"]:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

df[["is_top1pct_paid","peer_z_high","is_outlier_reimb","duplicate_suspect"]].sum()

is_top1pct_paid        6439
peer_z_high            2935
is_outlier_reimb     101896
duplicate_suspect       583
dtype: int64

##### Build intersection slices (claim-level “case packets”)

In [14]:
cols_core = ["claimid", "provider", "beneid", "claim_type", "claim_start", "claim_end",
             "reimb_amt", "deductible_paid", "los_days", "dx_count", "px_count", "drg",
             "z_ip_reimb", "z_op_reimb", "z_ip_los",
             "dup_exact_flag", "dup_near_count", "overcharge_z_flag", "overcharge_iqr_flag"]


def pick(cols):
    return [c for c in cols if c in df.columns]


pkt_short_and_top = df.loc[df["short_stay_costly"]
                           & df["is_top1pct_paid"], pick(cols_core)].copy()
pkt_dup_and_peer = df.loc[df["duplicate_suspect"] &
                          df["peer_z_high"],     pick(cols_core)].copy()
pkt_outlier_and_top = df.loc[df["is_outlier_reimb"]
                             & df["is_top1pct_paid"],  pick(cols_core)].copy()
pkt_overchg_and_top = df.loc[(df.get("overcharge_z_flag", 0).eq(1) | df.get(
    "overcharge_iqr_flag", 0).eq(1)) & df["is_top1pct_paid"], pick(cols_core)].copy()

# Label each packet with a reason (for triage)


def tag(d, label):
    x = d.copy()
    x["reason"] = label
    return x


claims_queue = pd.concat([
    tag(pkt_short_and_top,  "short_stay_high_zpay_ip ∩ top1pct"),
    tag(pkt_dup_and_peer,   "duplicate_suspect ∩ peer_z>3"),
    tag(pkt_outlier_and_top, "outlier(IQR|MAD) ∩ top1pct"),
    tag(pkt_overchg_and_top, "overcharge_rule ∩ top1pct")
], ignore_index=True).drop_duplicates(subset=["claimid", "reason"])

# Rank inside each reason by dollars (largest first)
claims_queue["rank_in_reason"] = claims_queue.groupby("reason")["reimb_amt"].rank(
    method="first", ascending=False)  # percentile ranks are below

claims_queue.head(100)

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,...,drg,z_ip_reimb,z_op_reimb,z_ip_los,dup_exact_flag,dup_near_count,overcharge_z_flag,overcharge_iqr_flag,reason,rank_in_reason
0,CLM69520,PRV52151,BENE74032,IP,2009-09-25,2009-09-26,"64,000.00","1,068.00",1.00,5,...,220,3.00,NaN,-0.82,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,6.00
1,CLM72294,PRV54504,BENE64896,IP,2009-10-17,2009-10-18,"57,000.00","1,068.00",1.00,9,...,294,3.00,NaN,-0.72,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,9.00
2,CLM78738,PRV51274,BENE131509,IP,2009-12-07,2009-12-08,"59,000.00","1,068.00",1.00,5,...,466,3.00,NaN,-1.09,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,8.00
3,CLM80589,PRV57103,BENE147111,IP,2009-12-23,2009-12-24,"57,000.00","1,068.00",1.00,9,...,292,3.00,NaN,-0.84,0,0,1,1,short_stay_high_zpay_ip ∩ top1pct,10.00
4,CLM51986,PRV53033,BENE69570,IP,2009-05-20,2009-05-21,"73,000.00","1,068.00",1.00,7,...,424,3.00,NaN,-0.88,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,4.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CLM669050,PRV51574,BENE153909,OP,2009-11-06,2009-11-06,"3,300.00",0.00,NaN,1,...,None,NaN,3.00,NaN,0,0,0,1,outlier(IQR|MAD) ∩ top1pct,"1,133.00"
96,CLM712674,PRV52120,BENE153921,OP,2009-12-03,2009-12-03,"7,700.00",0.00,NaN,1,...,None,NaN,3.00,NaN,0,0,0,1,outlier(IQR|MAD) ∩ top1pct,683.00
97,CLM608343,PRV57157,BENE15415,OP,2009-10-01,2009-10-01,"3,300.00",0.00,NaN,9,...,None,NaN,3.00,NaN,0,0,1,1,outlier(IQR|MAD) ∩ top1pct,"1,134.00"
98,CLM292311,PRV51347,BENE154447,OP,2009-04-08,2009-04-08,"3,300.00",0.00,NaN,3,...,None,NaN,3.00,NaN,0,0,1,1,outlier(IQR|MAD) ∩ top1pct,"1,135.00"


##### Provider-level triage table (counts, dollars, composite score)

In [15]:
# Build provider-level signals from the intersection claims
prov_agg = (claims_queue
            .groupby(["provider","reason"])
            .agg(n_claims=("claimid","count"),
                 dollars=("reimb_amt","sum"),
                 max_z_ip=("z_ip_reimb","max"),
                 max_z_op=("z_op_reimb","max"),
                 min_los=("z_ip_los","min"))   # min z_los => shorter than peers
            .reset_index())

# Pivot reasons to columns (counts + dollars per reason)
prov_pivot_counts = prov_agg.pivot_table(index="provider", columns="reason", values="n_claims", fill_value=0, aggfunc="sum")
prov_pivot_dollars = prov_agg.pivot_table(index="provider", columns="reason", values="dollars", fill_value=0.0, aggfunc="sum")

# Flatten MultiIndex columns with clearer prefixes
prov_pivot_counts.columns = [f"count_{c}" for c in prov_pivot_counts.columns]
prov_pivot_dollars.columns = [f"amt_{c}" for c in prov_pivot_dollars.columns]

# Join the two pivot tables
prov_sum = (prov_pivot_counts.join(prov_pivot_dollars, how="outer")
            .fillna(0)
            .reset_index())

# Add global totals (across all fraud reasons)
count_cols = [c for c in prov_sum.columns if c.startswith("count_")]
dollar_cols = [c for c in prov_sum.columns if c.startswith("amt_")]

prov_sum["total_flagged_claims"] = prov_sum[count_cols].sum(axis=1)
prov_sum["total_flagged_dollars"] = prov_sum[dollar_cols].sum(axis=1)

# Percentile ranks (0..1) per column for composite risk score
# Higher rank = higher risk
for c in count_cols + dollar_cols:
    prov_sum[f"rank_{c}"] = prov_sum[c].rank(pct=True, ascending=False)

# Calculate composite risk score (average of all percentile ranks)
rank_cols = [c for c in prov_sum.columns if c.startswith("rank_")]
prov_sum["composite_risk_score"] = prov_sum[rank_cols].mean(axis=1)

# Create audit queue: sort by risk score (highest first), then dollars, then count
audit_queue_providers = prov_sum.sort_values(
    ["composite_risk_score", "total_flagged_dollars", "total_flagged_claims"], 
    ascending=[False, False, False]
)

# Display top 20 highest-risk providers
audit_queue_providers.head(20)

,provider,count_duplicate_suspect ∩ peer_z>3,count_outlier(IQR|MAD) ∩ top1pct,count_overcharge_rule ∩ top1pct,count_short_stay_high_zpay_ip ∩ top1pct,amt_duplicate_suspect ∩ peer_z>3,amt_outlier(IQR|MAD) ∩ top1pct,amt_overcharge_rule ∩ top1pct,amt_short_stay_high_zpay_ip ∩ top1pct,total_flagged_claims,total_flagged_dollars,rank_count_duplicate_suspect ∩ peer_z>3,rank_count_outlier(IQR|MAD) ∩ top1pct,rank_count_overcharge_rule ∩ top1pct,rank_count_short_stay_high_zpay_ip ∩ top1pct,rank_amt_duplicate_suspect ∩ peer_z>3,rank_amt_outlier(IQR|MAD) ∩ top1pct,rank_amt_overcharge_rule ∩ top1pct,rank_amt_short_stay_high_zpay_ip ∩ top1pct,composite_risk_score
100,PRV51277,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
480,PRV52439,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
493,PRV52487,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
618,PRV52905,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
695,PRV53222,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
739,PRV53362,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
749,PRV53388,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
1105,PRV54681,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
1289,PRV55187,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
1421,PRV55513,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69


In [16]:
claims_queue.sort_values(["reason", "rank_in_reason"]).to_csv(
    "queue_claims_intersections.csv", index=False)
audit_queue_providers.to_csv("queue_providers_intersections.csv", index=False)

In [17]:
K = 100  # set this to your audit capacity
topK_providers = audit_queue_providers.head(K)["provider"].tolist()

# All intersection claims for those providers, prioritized by reason then amount
topK_claims = (claims_queue
               .loc[claims_queue["provider"].isin(topK_providers)]
               .sort_values(["reason", "reimb_amt"], ascending=[True, False]))

topK_providers_df = audit_queue_providers.head(K)
topK_claims.to_csv(f"queue_claims_top{K}.csv", index=False)
topK_providers_df.to_csv(f"queue_providers_top{K}.csv", index=False)

#### 3.5 - If possible, validate some rule-flagged cases manually or with domain expertise (e.g. check billing patterns).

A) Draw a stratified sample for SME review

In [18]:
# How many per reason? Adjust to your audit capacity.
target_per_reason = {
    "short_stay_high_zpay_ip ∩ top1pct": 50,
    "duplicate_suspect ∩ peer_z>3":      80,
    "overcharge_rule ∩ top1pct":         80,
    "outlier(IQR|MAD) ∩ top1pct":        90,
}

# Within each reason, sample with probability proportional to reimb_amt (prioritize $$)
samples = []
for reason, n in target_per_reason.items():
    block = claims_queue[claims_queue["reason"] == reason].copy()
    if block.empty:
        continue
    # PPS: probability proportional to size (reimb_amt); fall back to uniform if dollars missing
    w = block["reimb_amt"].clip(lower=1)
    p = w / w.sum()
    k = min(n, len(block))
    samp = block.sample(n=k, weights=p, replace=False, random_state=42)
    samp["review_batch"] = "batch_001"
    samples.append(samp)

review_sample = pd.concat(
    samples, ignore_index=True) if samples else pd.DataFrame()
review_sample.shape, review_sample["reason"].value_counts()

((190, 22),
 reason
 outlier(IQR|MAD) ∩ top1pct           90
 overcharge_rule ∩ top1pct            80
 short_stay_high_zpay_ip ∩ top1pct    16
 duplicate_suspect ∩ peer_z>3          4
 Name: count, dtype: int64)

B) Build the case-packet export for SMEs

In [ ]:
cols_for_review = [
    "claimid", "provider", "beneid", "claim_type", "claim_start", "claim_end",
    "reimb_amt", "deductible_paid", "los_days", "dx_count", "px_count", "drg",
    "z_ip_reimb", "z_ip_los", "z_op_reimb", "dup_exact_flag", "dup_near_count",
    "overcharge_z_flag", "overcharge_iqr_flag", "reason", "review_batch"
]
packet = review_sample[[
    c for c in cols_for_review if c in review_sample.columns]].copy()


def add_context(df):
    key = df[["provider", "beneid", "claim_start"]].copy()
    key["left"] = pd.to_datetime(key["claim_start"]) - pd.Timedelta(days=7)
    key["right"] = pd.to_datetime(key["claim_start"]) + pd.Timedelta(days=7)
    base = df.merge(
        df[["claimid", "provider", "beneid", "claim_start", "reimb_amt"]],
        on=["provider", "beneid"], suffixes=("", "_ctx")
    )
    base["claim_start"] = pd.to_datetime(base["claim_start"])
    base["claim_start_ctx"] = pd.to_datetime(base["claim_start_ctx"])
    mask = (base["claim_start_ctx"].between(base["claim_start"]-pd.Timedelta(days=7),
                                            base["claim_start"]+pd.Timedelta(days=7))) & \
        (base["claimid"] != base["claimid_ctx"])
    ctx = (base.loc[mask]
               .groupby("claimid")
               .agg(neighbor_claims=("claimid_ctx", "count"),
                    neighbor_paid=("reimb_amt_ctx", "sum"))
               .reset_index())
    return df.merge(ctx, on="claimid", how="left")


packet = add_context(packet).fillna({"neighbor_claims": 0, "neighbor_paid": 0})
packet.to_csv("review_requests_batch001.csv", index=False)

#### Generating the review_outcomes_batch001.csv

In [22]:
from pathlib import Path

def generate_outcomes_from_packet(packet: pd.DataFrame,
                                  out_path: str = "review_outcomes_batch001.csv",
                                  strict_shortstay: bool = False,
                                  zpay_thr: float = 3.0) -> pd.DataFrame:
    """
    Create SME outcomes CSV with conservative auto-fill:
      - Exact duplicate claims => disposition='Overpayment' (high confidence)
      - Short IP stay + high DRG peer z-pay => recommendation only (or Overpayment if strict_shortstay=True)
      - Peer 'overcharge' flags / top-1% pay => recommendation only
      - Everything carries a policy_basis string for SME audit trail
    """

    req_cols = ["claimid","provider","beneid","claim_type","claim_start","claim_end",
                "reimb_amt","los_days","drg","z_ip_reimb","z_op_reimb",
                "dup_exact_flag","dup_near_count","overcharge_z_flag","overcharge_iqr_flag",
                "reason"]
    missing = [c for c in req_cols if c not in packet.columns]
    if missing:
        raise ValueError(f"Packet missing columns: {missing}")

    out = packet.copy()

    # --- Base columns SME expects ---
    out["disposition"] = ""            # SME can overwrite; we prefill only clear duplicates
    out["note"] = ""                   # free-text from SME later

    # --- Helper booleans ---
    dup_exact = out["dup_exact_flag"].fillna(0).astype(int).eq(1)
    dup_near  = out["dup_near_count"].fillna(0).astype(int).gt(0)

    short_ip  = out["claim_type"].eq("IP") & out["los_days"].fillna(0).lt(2)
    high_zpay_ip = out["z_ip_reimb"].fillna(-np.inf).gt(zpay_thr)
    high_zpay_op = out["z_op_reimb"].fillna(-np.inf).gt(zpay_thr)

    overcharge = out.get("overcharge_z_flag", 0) == 1
    overcharge = overcharge | (out.get("overcharge_iqr_flag", 0) == 1)

    # --- Policy bases (concise) ---
    BASIS_DUP = ("Duplicate same-day service without appropriate modifier/units is denyable "
                 "(CMS coverage/processing guidance).")
    BASIS_2MN = ("Short inpatient stay + high DRG peer payment; validate Two-Midnight expectation "
                 "or exception (death/transfer/rapid improvement) per CMS guidance.")
    BASIS_PEER = ("Peer-normalized outlier payment; requires medical necessity & coding support "
                  "(Program Integrity medical review).")
    BASIS_NCCI = ("If units/codes exceed allowable combinations, check NCCI/MUE (when CPT/HCPCS present).")

    # --- Prefill exact duplicates as Overpayment (high confidence) ---
    out.loc[dup_exact, "disposition"] = "Overpayment"
    out.loc[dup_exact, "note"] = "Exact duplicate (same bene/provider/date/service); no proper modifier/units."

    # --- Auto recommendations (not binding) ---
    out["auto_recommendation"] = ""

    # Near duplicates -> review
    out.loc[dup_near & ~dup_exact, "auto_recommendation"] = "Review – possible duplicate (check modifiers/units)."

    # Short IP + high z-pay
    mask_short_high = short_ip & high_zpay_ip
    if strict_shortstay:
        out.loc[mask_short_high, "disposition"] = "Overpayment"
        out.loc[mask_short_high, "note"] = "Short stay with high DRG peer payment; no documented exception."
    else:
        out.loc[mask_short_high, "auto_recommendation"] = "Review – short IP stay + high DRG peer pay."

    # OP high z-pay (recommendation)
    out.loc[high_zpay_op & out["auto_recommendation"].eq(""), "auto_recommendation"] = \
        "Review – high outpatient peer-normalized payment."

    # Overcharge flags (recommendation)
    out.loc[overcharge & out["auto_recommendation"].eq(""), "auto_recommendation"] = \
        "Review – outlier payment vs peers / overcharge flag."

    # --- Policy basis column (multi-rule aware) ---
    out["policy_basis"] = ""
    out.loc[dup_exact | dup_near, "policy_basis"] = BASIS_DUP
    out.loc[mask_short_high, "policy_basis"] = (out.loc[mask_short_high, "policy_basis"] + " " + BASIS_2MN).str.strip()
    out.loc[(high_zpay_ip | high_zpay_op) & ~mask_short_high, "policy_basis"] = \
        (out.loc[(high_zpay_ip | high_zpay_op) & ~mask_short_high, "policy_basis"] + " " + BASIS_PEER).str.strip()

    # Mention NCCI/MUE as FYI for OP when codes/units are available
    out.loc[out["claim_type"].eq("OP") & (dup_near | overcharge), "policy_basis"] = \
        (out.loc[out["claim_type"].eq("OP") & (dup_near | overcharge), "policy_basis"] + " " + BASIS_NCCI).str.strip()

    # --- Keep only columns SME needs; extra helper columns are OK to keep too ---
    cols_export = ["claimid","disposition","note","auto_recommendation","policy_basis"]
    out_export = out[cols_export].copy()

    # Write CSV
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    out_export.to_csv(out_path, index=False)
    return out_export

In [25]:
# 1) Make sure you have the 'packet' DataFrame (from 3.5 sampling).
#    If not, reload it:
packet = pd.read_csv("review_requests_batch001.csv")

# 2) Paste/ensure the function definition is in the notebook,
#    then call it to create the CSV:

outcomes = generate_outcomes_from_packet(
    packet,
    out_path="review_outcomes_batch001.csv",
    # set True if you want short IP + high z-pay auto-marked Overpayment
    strict_shortstay=False,
    zpay_thr=3.0              # z-score cutoff you want to use for “high pay”
)

# 3) Sanity-check the file exists and looks right:
p = Path("review_outcomes_batch001.csv")
# True means it was written. :contentReference[oaicite:0]{index=0}
print("Exists?", p.exists())
check = pd.read_csv(p)
check.head(1000)

Exists? True


,claimid,disposition,note,auto_recommendation,policy_basis
0,CLM65415,NaN,NaN,Review – outlier payment vs peers / overcharge...,NaN
1,CLM59355,NaN,NaN,Review – short IP stay + high DRG peer pay.,Short inpatient stay + high DRG peer payment; ...
2,CLM38217,NaN,NaN,Review – short IP stay + high DRG peer pay.,Short inpatient stay + high DRG peer payment; ...
3,CLM53754,NaN,NaN,Review – outlier payment vs peers / overcharge...,NaN
4,CLM78738,NaN,NaN,Review – outlier payment vs peers / overcharge...,NaN
...,...,...,...,...,...
185,CLM59558,NaN,NaN,Review – outlier payment vs peers / overcharge...,NaN
186,CLM63643,NaN,NaN,NaN,NaN
187,CLM257200,NaN,NaN,Review – outlier payment vs peers / overcharge...,"If units/codes exceed allowable combinations, ..."
188,CLM54889,NaN,NaN,Review – outlier payment vs peers / overcharge...,NaN


C) Capture outcomes and compute PPV (Positive Predictive Value) by reason

In [26]:
# Load SME outcomes and compute yields
outcomes = pd.read_csv("review_outcomes_batch001.csv")
audited = packet.merge(outcomes, on="claimid", how="left")

# Positive predictive value (PPV) per reason
ppv = (audited.assign(pos=audited["disposition"].eq("Overpayment")))
ppv_by_reason = ppv.groupby("reason")["pos"].mean().sort_values(ascending=False).rename("PPV").reset_index()

# Dollars at stake by reason (optional)
by_reason = audited.groupby("reason")["reimb_amt"].sum().rename("total_paid").reset_index()

ppv_by_reason.merge(by_reason, on="reason", how="left").sort_values(["PPV","total_paid"], ascending=[False,False])


,reason,PPV,total_paid
0,duplicate_suspect ∩ peer_z>3,1.00,"50,660.00"
1,outlier(IQR|MAD) ∩ top1pct,0.00,"4,077,560.00"
2,overcharge_rule ∩ top1pct,0.00,"3,577,370.00"
3,short_stay_high_zpay_ip ∩ top1pct,0.00,"1,188,000.00"
